In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.backtest.backtest_ou import run_backtest

plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
df = pd.read_csv("../data/odds/MEX.csv")

print(df.shape)
df.head()

In [ ]:
# resultado real
df["goals_total"] = df["HG"] + df["AG"]

# ⚠️ TEMPORAL: odds simuladas
np.random.seed(42)
df["odds_over"] = np.random.uniform(1.8, 2.2, len(df))
df["odds_under"] = np.random.uniform(1.8, 2.2, len(df))

# placeholder λ
df["lambda_calibrated"] = 2.5

df.head()

In [ ]:
thresholds = {
    "min_ev": 0.03,
    "min_odds": 1.6,
    "max_odds": 3.2,
    "kelly_fraction": 0.25,
    "kelly_cap": 0.05,
}

In [ ]:
results, bets = run_backtest(df, thresholds)

print("RESULTS:")
print(results)

print("\nNúmero de bets:", len(bets))
bets.head()

In [ ]:
print(bets["ev"].describe())

bets["ev"].hist(bins=30)
plt.title("Distribución de EV")
plt.xlabel("EV")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
hit_rate = bets["win"].mean()
print("Hit rate:", round(hit_rate, 4))

In [ ]:
bets.groupby("side")["return"].mean()

In [ ]:
bets = bets.copy()
bets["cum_profit"] = bets["return"].cumsum()

bets["cum_profit"].plot()
plt.title("Curva de capital")
plt.xlabel("Número de apuestas")
plt.ylabel("Profit acumulado")
plt.show()

In [ ]:
ev_values = [0.01, 0.02, 0.03, 0.05, 0.08]

results_grid = []

for ev in ev_values:
    t = thresholds.copy()
    t["min_ev"] = ev

    res, _ = run_backtest(df, t)

    results_grid.append({
        "min_ev": ev,
        "roi": res["roi"],
        "n_bets": res["n_bets"],
        "avg_ev": res["avg_ev"],
    })

grid_df = pd.DataFrame(results_grid)
grid_df

In [ ]:
grid_df.plot(x="min_ev", y="roi", marker="o")
plt.title("ROI vs min_ev")
plt.show()

grid_df.plot(x="min_ev", y="n_bets", marker="o")
plt.title("Número de apuestas vs min_ev")
plt.show()

In [ ]:
plt.scatter(bets["ev"], bets["return"], alpha=0.3)
plt.xlabel("EV")
plt.ylabel("Return")
plt.title("EV vs Resultado")
plt.show()